# Modeling

## Chest X-Ray Pneumonia Detection


## Baseline CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.callbacks import TensorBoard
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)
import os

NUM_RUNS = 5

# ── Create models folder once ─────────────────────────────────────────────────
os.makedirs('models', exist_ok=True)

for run in range(1, NUM_RUNS + 1):
    print(f"\n{'='*60}")
    print(f"  TRAINING MODEL {run} OF {NUM_RUNS}")
    print(f"{'='*60}\n")

    # ── 1. Build model ────────────────────────────────────────────
    model = Sequential([
        Conv2D(16, (3, 3), 1, activation='relu', input_shape=(150, 150, 3)),
        MaxPooling2D(),
        Conv2D(32, (3, 3), 1, activation='relu'),
        MaxPooling2D(),
        Conv2D(16, (3, 3), 1, activation='relu'),
        MaxPooling2D(),
        Flatten(),
        Dense(256, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss=tf.losses.BinaryCrossentropy(),
        metrics=['accuracy']
    )

    # ── 2. Train ──────────────────────────────────────────────────
    logdir = f'logs/run_{run}'
    os.makedirs(logdir, exist_ok=True)
    tensorboard_callback = TensorBoard(log_dir=logdir)

    hist = model.fit(
        train_generator,
        epochs=20,
        validation_data=val_generator,
        callbacks=[tensorboard_callback]
    )

    model_path = f'models/model_run_{run}.h5'   # ← saved inside models/
    model.save(model_path)
    print(f"\nModel saved → {model_path}")

    # ── 3. Evaluate ───────────────────────────────────────────────
    best_model = load_model(model_path)

    val_loss, val_acc = best_model.evaluate(val_generator, verbose=0)
    print(f"\nVal   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    test_loss, test_acc = best_model.evaluate(test_generator, verbose=0)
    print(f"Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.4f}")

    # ── 4. Predictions ────────────────────────────────────────────
    test_generator.reset()
    y_pred_probs = best_model.predict(test_generator, verbose=1).flatten()
    y_pred       = (y_pred_probs > 0.5).astype(int)
    y_true       = test_generator.classes[:len(y_pred)]

    # ── 5. Classification report ──────────────────────────────────
    print(f"\n── Classification Report  (Run {run}) ──")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Pneumonia']))

    # ── 6. All plots in one figure (2 rows × 3 cols) ──────────────
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle(f'Model Run {run} — Evaluation Dashboard', fontsize=15, fontweight='bold')

    # ── 6a. Confusion Matrix ──────────────────────────────────────
    ax = axes[0, 0]
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Pneumonia'],
                yticklabels=['Normal', 'Pneumonia'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title('Confusion Matrix')

    # ── 6b. Training Curves — Loss ────────────────────────────────
    ax = axes[0, 1]
    ax.plot(hist.history['loss'],     label='Train loss', color='teal')
    ax.plot(hist.history['val_loss'], label='Val loss',   color='orange')
    ax.set_title('Training & Validation Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)

    # ── 6c. Training Curves — Accuracy ───────────────────────────
    ax = axes[0, 2]
    ax.plot(hist.history['accuracy'],     label='Train acc', color='teal')
    ax.plot(hist.history['val_accuracy'], label='Val acc',   color='orange')
    ax.set_title('Training & Validation Accuracy')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)

    # ── 6d. ROC Curve ─────────────────────────────────────────────
    ax = axes[1, 0]
    fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_title('ROC Curve')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.legend(); ax.grid(alpha=0.3)

    # ── 6e. Prediction Confidence Distribution ────────────────────
    ax = axes[1, 1]
    ax.hist(y_pred_probs[y_true == 0], bins=30, alpha=0.6,
            color='steelblue', label='Normal')
    ax.hist(y_pred_probs[y_true == 1], bins=30, alpha=0.6,
            color='tomato',    label='Pneumonia')
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='Threshold = 0.5')
    ax.set_title('Prediction Confidence Distribution')
    ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Count')
    ax.legend(); ax.grid(alpha=0.3)

    # ── 6f. Precision–Recall Curve ────────────────────────────────
    ax = axes[1, 2]
    precision, recall, _ = precision_recall_curve(y_true, y_pred_probs)
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, color='purple', lw=2, label=f'PR AUC = {pr_auc:.3f}')
    ax.set_title('Precision–Recall Curve')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plot_path = f'evaluation_run_{run}.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Plots saved → {plot_path}")

print("\n✅ All 5 runs complete.")

# Model 1 — Baseline CNN
## Chest X-Ray Pneumonia Detection

---

## 1. Overview

Here i cover the training, evaluation, and analysis of the first model in this study — a simple Convolutional Neural Network (CNN) built entirely from scratch. This model serves as the **baseline** against which all other models are compared. It uses no pretrained weights, no external knowledge, and no advanced architecture. Everything it learns comes entirely from the 4434 training images provided.

---

## 2. Architecture

The model is a Sequential CNN with three convolutional blocks followed by a fully connected classification head.

| Layer | Details |
|---|---|
| Conv2D | 16 filters, 3×3 kernel, stride 1, ReLU activation |
| MaxPooling2D | Default 2×2 pool size |
| Conv2D | 32 filters, 3×3 kernel, stride 1, ReLU activation |
| MaxPooling2D | Default 2×2 pool size |
| Conv2D | 16 filters, 3×3 kernel, stride 1, ReLU activation |
| MaxPooling2D | Default 2×2 pool size |
| Flatten | Converts spatial feature maps to a 1D vector |
| Dense | 256 units, ReLU activation |
| Dense (output) | 1 unit, Sigmoid activation |

The output layer produces a single probability between 0 and 1. A threshold of **0.5** is used as the decision boundary — values above 0.5 are classified as Pneumonia, values below as Normal.

The model was compiled with:
- **Optimizer:** Adam (default learning rate 1e-3)
- **Loss:** Binary Crossentropy
- **Metrics:** Accuracy

---

## 3. Training

The model was trained for **20 epochs** with no callbacks and no regularization. This decision was made after i experimented with callbacks and regularization in separate experiments, all of which performed worse than this simple baseline. The conclusion i made was that for this dataset and architecture, callbacks and regularization were over-constraining a model that was not severely overfitting to begin with.

### Training Curves

![Training Loss and Accuracy Curves](documentation/baseline_loss_curves(1).png)

*Figure 1: Training Loss and Accuracy Curves — Run 2*




The loss curves show healthy learning, both training and validation loss decline together. A mild divergence appears where training loss continues declining while validation loss becomes noisy and spikes slightly. This is a mild overfitting signature but it appeared late and did not significantly impact test performance.

---

## 4. Training Stability — 5 Independent Runs

Because CNNs stars from **random weight initialization** every time they are trained, results vary between runs. To properly characterize this variance, the baseline model, and all the models of the study, were trained 5 independent times with everything held constant — same architecture, same data, same augmentation, same epochs — with only the random weight initialization differing.

| Run | Accuracy | Normal Precision | Normal Recall | Pneumonia Recall | Normal F1 | Pneumonia F1 |
|---|---|---|---|---|---|---|
| Run 1 | 0.76 | 0.93 | 0.40 | 0.98 | 0.56 | 0.84 |
| Run 2 | 0.87 | 0.93 | 0.70 | 0.97 | 0.80 | 0.90 |
| Run 3 | 0.83 | 0.97 | 0.55 | 0.99 | 0.70 | 0.88 |
| Run 4 | 0.78 | 0.98 | 0.43 | 0.99 | 0.60 | 0.85 |
| Run 5 | 0.84 | 0.95 | 0.59 | 0.98 | 0.73 | 0.88 |

### Aggregate Statistics

| Metric | Mean | Median | Std | Min | Max |
|---|---|---|---|---|---|
| Accuracy | 0.816 | 0.830 | 0.042 | 0.76 | 0.87 |
| Normal Precision | 0.952 | 0.950 | 0.022 | 0.93 | 0.98 |
| Normal Recall | 0.534 | 0.550 | 0.113 | 0.40 | 0.70 |
| Pneumonia Recall | 0.982 | 0.980 | 0.008 | 0.97 | 0.99 |
| Normal F1 | 0.678 | 0.700 | 0.093 | 0.56 | 0.80 |
| Pneumonia F1 | 0.870 | 0.880 | 0.023 | 0.84 | 0.90 |

### What the standard deviation tells us

The standard deviation column is the most important part of this table. It reveals a fundamental asymmetry in how reliably this model learns each class.

**Pneumonia Recall std = 0.008 — rock solid**

Every single run catches between 97% and 99% of Pneumonia cases. This number barely moves regardless of initialization. The reason is the class imbalance — the training set contains 2.9 times more Pneumonia cases than Normal cases. Every batch delivers roughly 3 times more gradient signal from Pneumonia than Normal. By early stages every model has already learned Pneumonia features reliably regardless of where the weights started.

**Normal Recall std = 0.113 — highly unstable**

Normal Recall swings from 0.40 to 0.70 across runs — a range of 30 percentage points. This means in any given training run we have no way of knowing whether the model will correctly identify 40% or 70% of healthy patients. The outcome depends entirely on the luck of weight initialization.

After learning Pneumonia quickly, the remaining training capacity is spent trying to distinguish Normal from Pneumonia in ambiguous cases. This is where initialization matters. Some random starting points lead the model to sensible decision boundaries for Normal cases. Others lead it toward weight configurations that are overly aggressive toward Pneumonia for ambiguous inputs. There is no mechanism in this training setup to control which path the model takes.

**Normal Precision std = 0.022 — stable**

Normal Precision stays between 0.93 and 0.98 across all runs. When the model does predict Normal, it is consistently correct around 95% of the time. The model is conservative about Normal predictions — it only makes the Normal call when fairly certain — which is why precision is stable and high even when recall is poor.

---

## 5. Detailed Evaluation — Best Run (Run 2)

The following detailed evaluation is based on **Run 2**, the best performing run across all 5 iterations, achieving the highest accuracy (0.87), highest Normal Recall (0.70), and the best overall balance between both classes.
 
### Classification Report (Run 2)
 
```
              precision    recall  f1-score   support
 
      Normal       0.93      0.70      0.80       234
   Pneumonia       0.84      0.97      0.90       390
 
    accuracy                           0.87       624
   macro avg       0.89      0.83      0.85       624
weighted avg       0.88      0.87      0.86       624
```
 
**Pneumonia Recall (0.97):** The model correctly identifies 97 out of every 100 Pneumonia patients. This is the most clinically important metric — missing a sick patient is the most dangerous error. 0.97 is excellent.
 
**Normal Recall (0.70):** The model incorrectly flags 30% of healthy patients as Pneumonia. This is the model's primary weakness. While less dangerous than missing Pneumonia, it creates unnecessary burden — false alarms lead to further testing and patient anxiety. Critically, this number is highly variable across runs (std=0.113), meaning this result cannot be reliably reproduced.
 
**Normal Precision (0.93):** When the model does predict Normal, it is correct 93% of the time. The model is conservative — it only predicts Normal when fairly certain, which is why precision is stable but recall is variable.
 
### Confusion Matrix
 
![Confusion Matrix](documentation/confusion_matrix_baseline_best.png)
*Figure 2: Confusion matrix — Run 2 (best run)*
 
| | Predicted Normal | Predicted Pneumonia |
|---|---|---|
| **Actual Normal** | 163  True Negative | 71  False Positive |
| **Actual Pneumonia** | 12  False Negative | 378  True Positive |
 
The **12 False Negatives** represent Pneumonia patients the model missed entirely — the most dangerous error in a clinical context. 12 out of 390 is consistent with the 97% Pneumonia Recall. The **71 False Positives** represent healthy patients incorrectly flagged as sick. These patients would be referred for unnecessary further examination.
 
### ROC Curve
 
![ROC Curve](documentation/baseline_roc_curve_best.png)
*Figure 3: ROC Curve — Run 2 (AUC = 0.949)*
 
An AUC of **0.949** is the starting point for comparison across all three models in this study:
 
```
Baseline:     AUC = 0.949
EfficientNet: AUC = 0.921  worse than baseline
DenseNet:     AUC = 0.974  better than baseline
```
 
This is something that needs to be pointed out. EfficientNet, despite having 5 million pretrained parameters, achieves a lower AUC than the simple baseline CNN. DenseNet is the only model that meaningfully improves on the baseline's discriminative ability.
 
The curve hugs the top-left corner tightly, particularly in the early portion. At a False Positive Rate of 0.10, the model achieves approximately 0.90 True Positive Rate — catching 90% of sick patients while only incorrectly flagging 10% of healthy ones. This is a strong operating point for a model with no pretrained knowledge.
 
### Precision-Recall Curve
 
![Precision-Recall Curve](documentation/precision_recall_curve_baseline_best.png)
*Figure 4: Precision-Recall Curve — Run 2 (PR AUC = 0.966)*
 
A PR AUC of **0.966** is strong and again outperforms EfficientNet (0.940):
 
```
Baseline:     PR AUC = 0.966
EfficientNet: PR AUC = 0.940  ← worse than baseline
DenseNet:     PR AUC = 0.983  ← better than baseline
```
 
The curve maintains near-perfect precision from recall 0 to approximately 0.80, only beginning to degrade noticeably above 0.85 recall. This means the baseline can push Pneumonia detection to 80% recall while essentially generating no false alarms. The degradation pattern is slightly earlier than DenseNet (which holds near-perfect precision to 0.80 recall) but significantly better than EfficientNet (which begins degrading around 0.60 recall).
 
The strong PR AUC of 0.966 confirms that despite the class imbalance and lack of pretraining, the baseline has genuinely learned to detect Pneumonia well. Its weakness is not Pneumonia detection — it is the unreliable Normal detection caused by initialization variance.
 
### Prediction Confidence Distribution
 
![Confidence Distribution](documentation/baseline_confidence_dist_best.png)
*Figure 5: Prediction confidence distribution — Run 2*
 
The distribution reveals the model's internal confidence profile directly:
 
**Left of threshold (predicted Normal):**
- A cluster of approximately 47 Normal cases near 0 — the model is confident about clearly Normal cases
- A flat, spreading tail of Normal cases extending all the way from 0.1 to past 0.5 — these are the borderline Normal cases the model is uncertain about
- Only a handful of Pneumonia cases fall below the threshold — confirming the strong Pneumonia Recall
**Right of threshold (predicted Pneumonia):**
- A dominant spike of approximately 310 Pneumonia cases concentrated near 1.0 — the model is extremely confident about Pneumonia predictions
- A meaningful number of Normal cases spread across the 0.5–1.0 range — the 71 false positives
The shape of the Normal distribution is the most revealing part of this plot. Unlike DenseNet which shows a tall tight Normal cluster near 0, the baseline shows a much shorter initial spike that quickly flattens into a long tail. This flat tail extending from 0.1 all the way past the decision threshold represents Normal cases the model has no confident view on — and for all of them it defaults to Pneumonia.
 
This visual directly explains why Normal Recall is only 0.70 even in the best run. The model confidently identifies the clearly Normal cases but is genuinely uncertain about a large proportion of Normal cases, resolving that uncertainty by defaulting to the majority class. This behavior is the visual signature of class imbalance operating on a model with no prior knowledge of what Normal lung tissue actually looks like.

---

## 6. Understanding the Precision-Recall Tradeoff

The following pattern appears consistently across all 5 runs:

```
Normal Precision:  high and stable  (~0.93–0.98)
Normal Recall:     low and variable (0.40–0.70)
```

The model is conservative about predicting Normal. It only does so when it sees strong evidence. When it does say Normal, it is almost always right (high precision). But it misses many actual Normal cases that fall anywhere near the decision boundary (low recall).

This is a direct consequence of the class imbalance. Because the model sees 2.9x more Pneumonia than Normal during training, it develops a strong internal prior that any given patient is probably Pneumonia. It therefore defaults to Pneumonia for anything ambiguous.

---

## 7. Summary

| Finding | Detail |
|---|---|
| Mean test accuracy | 81.6% across 5 runs |
| Normal Precision | 95.2% mean — stable (std=0.022) |
| Normal Recall | 53.4% mean — highly unstable (std=0.113) |
| Pneumonia Recall | 98.2% mean — rock solid (std=0.008) |
| Core weakness | Class imbalance drives Pneumonia bias |
| Core limitation | Results not reproducible due to random initialization |
| Best single run | Run 2: 87% accuracy, 70% Normal Recall, 97% Pneumonia Recall |

The baseline proves that the pneumonia detection task is learnable with minimal resources. Its fundamental limitation is not architectural complexity but data imbalance and initialization instability — problems that more sophisticated models address in different ways, as documented in the subsequent model reports.

## EfficientNet-B0

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)

# ── Config ────────────────────────────────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 32
SEED       = 42
TRAIN_DIR  = 'data/train'
TEST_DIR   = 'data/test'
NUM_RUNS   = 5

os.makedirs('models', exist_ok=True)

# ── Data generators (built once — reused every run) ───────────────────────────
train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    horizontal_flip        = True,
    rotation_range         = 10,
    zoom_range             = 0.1,
    width_shift_range      = 0.1,
    height_shift_range     = 0.1,
    brightness_range       = [0.8, 1.2],
    validation_split       = 0.15
)
val_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    validation_split       = 0.15
)
test_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', subset='training', shuffle=True, seed=SEED
)
val_generator = val_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', subset='validation', shuffle=False, seed=SEED
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)

# ── 5 Runs ────────────────────────────────────────────────────────────────────
for run in range(1, NUM_RUNS + 1):
    print(f"\n{'='*60}")
    print(f"  RUN {run} OF {NUM_RUNS}")
    print(f"{'='*60}\n")

    phase1_path = f'models/efficientnet_phase1_run{run}.h5'
    phase2_path = f'models/efficientnet_phase2_run{run}.h5'

    # ── Build model ───────────────────────────────────────────────
    base_model = EfficientNetB0(
        weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base_model.trainable = False

    inputs  = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base_model(inputs, training=False)
    x       = GlobalAveragePooling2D()(x)
    x       = BatchNormalization()(x)
    x       = Dense(128, activation='relu')(x)
    x       = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    model_efficientnet = Model(inputs, outputs)

    # ── Phase 1: Feature Extraction ───────────────────────────────
    print(f"\n── Phase 1: Feature Extraction  (Run {run}) ──")
    model_efficientnet.compile(
        optimizer = tf.keras.optimizers.Adam(1e-3),
        loss      = tf.keras.losses.BinaryCrossentropy(),
        metrics   = ['accuracy']
    )
    checkpoint_phase1 = ModelCheckpoint(
        phase1_path, monitor='val_loss',
        save_best_only=True, verbose=1
    )
    hist_phase1 = model_efficientnet.fit(
        train_generator, epochs=15,
        validation_data=val_generator,
        callbacks=[checkpoint_phase1]
    )

    # ── Phase 2: Fine-Tuning ──────────────────────────────────────
    print(f"\n── Phase 2: Fine-Tuning  (Run {run}) ──")
    model_efficientnet.load_weights(phase1_path)

    unfreeze_from = int(len(base_model.layers) * 0.8)
    base_model.trainable = True
    for layer in base_model.layers[:unfreeze_from]:
        layer.trainable = False

    model_efficientnet.compile(
        optimizer = tf.keras.optimizers.Adam(1e-5),
        loss      = tf.keras.losses.BinaryCrossentropy(),
        metrics   = ['accuracy']
    )
    checkpoint_phase2 = ModelCheckpoint(
        phase2_path, monitor='val_loss',
        save_best_only=True, verbose=1
    )
    early_stop = EarlyStopping(
        monitor='val_loss', patience=7,
        restore_best_weights=True, verbose=1
    )
    hist_phase2 = model_efficientnet.fit(
        train_generator, epochs=30,
        validation_data=val_generator,
        callbacks=[checkpoint_phase2, early_stop]
    )

    # ── Load best phase-2 weights & evaluate ──────────────────────
    best_model = load_model(phase2_path)

    val_loss, val_acc   = best_model.evaluate(val_generator,  verbose=0)
    test_loss, test_acc = best_model.evaluate(test_generator, verbose=0)
    print(f"\nVal  — Loss: {val_loss:.4f}  |  Acc: {val_acc:.4f}")
    print(f"Test — Loss: {test_loss:.4f}  |  Acc: {test_acc:.4f}")

    # ── Predictions ───────────────────────────────────────────────
    test_generator.reset()
    y_pred_probs = best_model.predict(test_generator, verbose=1).flatten()
    y_pred       = (y_pred_probs > 0.5).astype(int)
    y_true       = test_generator.classes[:len(y_pred)]

    # ── Classification report ─────────────────────────────────────
    print(f"\n── Classification Report  (Run {run}) ──")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Pneumonia']))

    # ── Evaluation dashboard ──────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle(
        f'EfficientNet-B0  Run {run} — Evaluation Dashboard',
        fontsize=15, fontweight='bold'
    )

    # 1. Confusion Matrix
    ax = axes[0, 0]
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Pneumonia'],
                yticklabels=['Normal', 'Pneumonia'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title('Confusion Matrix')

    # 2. Phase 2 Loss curves
    ax = axes[0, 1]
    ax.plot(hist_phase2.history['loss'],     label='Train loss', color='teal')
    ax.plot(hist_phase2.history['val_loss'], label='Val loss',   color='orange')
    ax.set_title('Phase 2 — Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)

    # 3. Phase 2 Accuracy curves
    ax = axes[0, 2]
    ax.plot(hist_phase2.history['accuracy'],     label='Train acc', color='teal')
    ax.plot(hist_phase2.history['val_accuracy'], label='Val acc',   color='orange')
    ax.set_title('Phase 2 — Accuracy')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)

    # 4. ROC Curve
    ax = axes[1, 0]
    fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_title('ROC Curve')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.legend(); ax.grid(alpha=0.3)

    # 5. Prediction Confidence Distribution
    ax = axes[1, 1]
    ax.hist(y_pred_probs[y_true == 0], bins=30, alpha=0.6,
            color='steelblue', label='Normal')
    ax.hist(y_pred_probs[y_true == 1], bins=30, alpha=0.6,
            color='tomato',    label='Pneumonia')
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='Threshold = 0.5')
    ax.set_title('Prediction Confidence Distribution')
    ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Count')
    ax.legend(); ax.grid(alpha=0.3)

    # 6. Precision–Recall Curve
    ax = axes[1, 2]
    precision, recall, _ = precision_recall_curve(y_true, y_pred_probs)
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, color='purple', lw=2, label=f'PR AUC = {pr_auc:.3f}')
    ax.set_title('Precision–Recall Curve')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plot_path = f'evaluation_efficientnet_run{run}.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Plots saved → {plot_path}")

print("\n✅ All 5 EfficientNet runs complete.")

# Model 2 — EfficientNet-B0 (Fine-Tuned)
## Chest X-Ray Pneumonia Detection

---

## 1. Overview

Here i cover the training, evaluation, and analysis of the second model — a fine-tuned **EfficientNet-B0** pretrained on ImageNet. This model represents the first transfer learning experiment in the study. Unlike the baseline CNN which started from random weights, EfficientNet-B0 begins with knowledge learned from 1.4 million natural images across 1000 categories.

The central question this model answers is: **does general visual knowledge from natural photos transfer meaningfully to chest X-ray diagnosis?**

The answer, as the results show, is: **no — not meaningfully enough**. Despite the pretrained weights, EfficientNet-B0 underperforms the baseline CNN on almost every metric. More importantly, it exhibits a severe and consistent Pneumonia prediction bias that makes it the least clinically reliable model in this study. Understanding why this happens is one of the most important lessons the study produces.

---

## 2. What is EfficientNet-B0?

EfficientNet-B0 is the smallest member of the EfficientNet family, developed by Google in 2019. It was designed around a principle called **compound scaling** — simultaneously scaling the depth (number of layers), width (number of filters per layer), and resolution (input image size) in a mathematically optimal ratio.

| Model | Parameters | ImageNet Accuracy |
|---|---|---|
| VGG16 | 138 million | 71.3% |
| ResNet50 | 25 million | 76.0% |
| EfficientNet-B0 | 5 million | 77.1% |

EfficientNet-B0 achieves better ImageNet accuracy than ResNet50 with 5x fewer parameters. On paper this looks ideal for a small dataset. In practice, the domain gap between ImageNet and chest X-rays undermined this advantage entirely.

### How EfficientNet processes information

EfficientNet uses a sequential feature transformation architecture:

```
Input image
    ↓
Early layers  → detect edges, corners, color gradients
    ↓           (information transformed and passed forward)
Middle layers → detect textures and simple shapes
    ↓           (early features no longer directly accessible)
Deep layers   → detect complex patterns
    ↓           (only high-level abstractions remain)
Global Average Pooling
    ↓
Classification head
```

Each layer receives input only from the layer immediately before it. Information is progressively abstracted and the fine-grained low-level features from early layers are no longer directly available to deep layers. This architectural property is central to understanding why EfficientNet struggles on chest X-rays.

---

## 3. Transfer Learning — Two-Phase Approach

### Phase 1 — Feature Extraction (15 epochs)

The entire EfficientNet-B0 backbone is frozen. Only the new classification head is trained.

```
EfficientNet-B0 backbone → FROZEN
Classification head       → TRAINED from scratch
Learning rate: 1e-3
Callbacks: ModelCheckpoint (save best val_loss)
```

### Phase 2 — Fine-Tuning (up to 30 epochs)

The last 20% of backbone layers are unfrozen and the entire model is trained with a very small learning rate.

```
First 80% of backbone → FROZEN
Last 20% of backbone  → UNFROZEN
Classification head   → CONTINUES training
Learning rate: 1e-5
Callbacks: ModelCheckpoint + EarlyStopping(patience=7)
```

### Critical preprocessing detail

EfficientNet-B0 requires its own preprocessing function — not `rescale=1./255`:

```python
from tensorflow.keras.applications.efficientnet import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input, ...
)
```

Using `rescale=1./255` caused the very first EfficientNet attempt to completely collapse — predicting every single image as Pneumonia with 0% Normal Recall. Correct preprocessing is mandatory.

---

## 4. Architecture and Classification Head

```
Input: 224 × 224 × 3
    ↓
EfficientNet-B0 backbone (pretrained on ImageNet)
    ↓
GlobalAveragePooling2D
    ↓
BatchNormalization
    ↓
Dense(128, relu)
    ↓
Dropout(0.3)
    ↓
Dense(1, sigmoid)
```

The same classification head used for DenseNet-121 was kept identical to ensure a fair comparison. Any performance difference between the two transfer learning models comes from the backbone architecture, not the head.

---

## 5. Training Stability — 5 Independent Runs

| Run | Accuracy | Normal Precision | Normal Recall | Pneumonia Recall | Normal F1 | Pneumonia F1 |
|---|---|---|---|---|---|---|
| Run 1 | 0.80 | 0.97 | 0.50 | 0.99 | 0.66 | 0.86 |
| Run 2 | 0.76 | 0.98 | 0.38 | 0.99 | 0.55 | 0.84 |
| Run 3 | 0.81 | 0.96 | 0.51 | 0.99 | 0.66 | 0.87 |
| Run 4 | 0.74 | 0.99 | 0.32 | 1.00 | 0.48 | 0.83 |
| Run 5 | 0.83 | 0.97 | 0.57 | 0.99 | 0.72 | 0.88 |

### Aggregate Statistics

| Metric | Mean | Median | Std | Min | Max |
|---|---|---|---|---|---|
| Accuracy | 0.788 | 0.800 | 0.033 | 0.74 | 0.83 |
| Normal Precision | 0.974 | 0.970 | 0.011 | 0.96 | 0.99 |
| Normal Recall | 0.456 | 0.500 | 0.093 | 0.32 | 0.57 |
| Pneumonia Recall | 0.992 | 0.990 | 0.004 | 0.99 | 1.00 |
| Normal F1 | 0.614 | 0.660 | 0.088 | 0.48 | 0.72 |
| Pneumonia F1 | 0.856 | 0.860 | 0.019 | 0.83 | 0.88 |

---

## 6. The Core Problem — Severe and Consistent Pneumonia Bias

EfficientNet's results are not just weak — they reveal a specific and serious behavioral problem that makes this model clinically unreliable.

### The pattern across all 5 runs

```
Pneumonia Recall: 0.99, 0.99, 0.99, 1.00, 0.99  ← almost always 0.99+
Normal Recall:    0.50, 0.38, 0.51, 0.32, 0.57  ← wildly inconsistent
```

Pneumonia Recall is nearly identical across every single run. Four runs give 0.99 and Run 4 gives a perfect 1.00. This looks impressive until we look at what is really happening.

### Why near-perfect Pneumonia Recall is a red flag here

A Pneumonia Recall of 0.99–1.00 sounds clinically ideal. But the way this model achieves it is not through good learning — it is through extreme bias. The model has learned to **predict Pneumonia for almost everything**.

Consider Run 4, the most extreme case:

```
Run 4:
Normal Precision:  0.99  ← when it says Normal, it is almost always right
Normal Recall:     0.32  ← but it only says Normal for 32% of Normal patients
Pneumonia Recall:  1.00  ← catches every single Pneumonia patient
```

The only way to guarantee 100% Pneumonia Recall is to never predict Normal for anything ambiguous. Run 4's model is so aggressive toward Pneumonia that it flags 68% of healthy patients as sick, just to ensure it never misses a single Pneumonia case. This is not a good model. It is a heavily biased one that has essentially given up on Normal detection in exchange for perfect Pneumonia detection.

### The precision-recall contradiction

The most revealing pattern is the extreme contrast between Normal Precision and Normal Recall across all runs:

```
Normal Precision: 0.974 mean  ← near perfect, almost never wrong when it says Normal
Normal Recall:    0.456 mean  ← misses 54% of all actual Normal patients
```

These two numbers together describe exactly what is happening. The model is extremely conservative about predicting Normal. It only makes the Normal call when the evidence is overwhelming — which is why precision is near-perfect. But 54% of all actual Normal patients do not clear that threshold, so they get misclassified as Pneumonia.


### Why does EfficientNet show this bias more severely than the baseline?

Both the baseline and EfficientNet are affected by the class imbalance. But EfficientNet's bias is more consistent and more severe. The reason is architectural.

EfficientNet's sequential feature transformation means that by the time information reaches the final classification layers, the fine-grained textural features that distinguish Normal lung tissue from Pneumonia consolidation have been progressively compressed and abstracted away. The model ends up with strong representations for Pneumonia — which has more obvious structural changes that survive the progressive abstraction — but weak and uncertain representations for Normal lung tissue — which requires preserving subtle low-level textural details that get lost during sequential transformation.

When the model is uncertain about a case (which is most Normal cases and borderline cases), it falls back on its training prior — which says Pneumonia is 2.9x more likely. The result is a model that is confidently correct about Pneumonia and systematically uncertain about Normal.

The baseline, despite having no pretrained features, shows slightly better Normal Recall on average (0.534 vs 0.456) precisely because it does not have this systematic architectural bias. Its uncertainty is random rather than directional.

### The clinical implication

Across all 5 runs, EfficientNet's Normal Recall ranged from **0.32 to 0.57**. This means in a clinical deployment you could get a model that correctly identifies only 32% of healthy patients — flagging 68% for unnecessary further examination. For a medical screening tool this is unacceptable regardless of how good the Pneumonia Recall is.

---

## 7. EfficientNet vs Baseline — The Honest Comparison

With 5 runs each the comparison is clear and unfavorable for EfficientNet:

| Metric | Baseline Mean | Baseline Std | EfficientNet Mean | EfficientNet Std |
|---|---|---|---|---|
| Accuracy | **0.816** | ±0.042 | 0.788 | ±0.033 |
| Normal Recall | **0.534** | ±0.113 | 0.456 | ±0.093 |
| Pneumonia Recall | 0.982 | ±0.008 | **0.992** | ±0.004 |
| Normal F1 | **0.678** | ±0.093 | 0.614 | ±0.088 |
| Pneumonia F1 | **0.870** | ±0.023 | 0.856 | ±0.019 |

EfficientNet is worse than the baseline on accuracy, Normal Recall, Normal F1, and Pneumonia F1. It is only better on Pneumonia Recall — and as established above, this improvement is achieved through bias rather than genuine learning.

The one genuine advantage EfficientNet shows is slightly lower standard deviation on some metrics — the pretrained weights do provide a more consistent starting point. But consistency in the wrong direction is not a clinical advantage.

---

## 8. Detailed Evaluation — Best Run (Run 5)
 
The following evaluation is based on **Run 5**, the best performing run (Accuracy 0.83, Normal Recall 0.57, Pneumonia Recall 0.99).
 
### Classification Report (Run 5)
 
```
              precision    recall  f1-score   support
 
      Normal       0.97      0.57      0.72       234
   Pneumonia       0.79      0.99      0.88       390
 
    accuracy                           0.83       624
   macro avg       0.88      0.78      0.80       624
weighted avg       0.86      0.83      0.82       624
```
 
Even in the best run, the model only correctly identifies 57% of healthy patients. Normal Precision of 0.97 confirms the conservative behavior — when it does predict Normal it is almost always right, but it makes that prediction for barely more than half of all actual Normal cases.
 
### Confusion Matrix
 
![Confusion Matrix](documentation/confusion_matrix_efficient_best.png)
*Figure 1: Confusion matrix — Run 5 (best run)*
 
| | Predicted Normal | Predicted Pneumonia |
|---|---|---|
| **Actual Normal** | 133  True Negative | 101  False Positive |
| **Actual Pneumonia** | 4  False Negative | 386  True Positive |
 
The **4 False Negatives** are the most critical number — only 4 Pneumonia patients out of 390 were missed. However the **101 False Positives** are deeply concerning — 101 healthy patients were incorrectly flagged as sick. This means 43.2% of all Normal patients were sent for unnecessary further examination. This is the most direct illustration of EfficientNet's Pneumonia bias — it is so aggressive toward Pneumonia that it sacrifices over 100 healthy patients per 234 to avoid missing a single sick one.
 
### Training Curves — Phase 2
 
![Phase 2 Loss and Accuracy](documentation/loss_curves_efficient_best.png)
*Figure 2: Phase 2 training and validation loss curves — Run 5*
 
The Phase 2 loss curves ran for approximately **30 epochs** — EarlyStopping with patience=7 allowed training to continue for a long time before halting:
 
```
Train loss:  0.19 → 0.05  (declining consistently throughout)
Val loss:    0.13 → 0.09  (declining but noisily, with spikes)
```
 
Both losses are declining which is a sign of genuine learning throughout Phase 2. However the gap between training loss (ending ~0.05) and validation loss (ending ~0.09) is widening in the later epochs — a mild overfitting signature. The validation loss is also significantly noisier than the training loss, oscillating rather than smoothly declining. This instability in the validation curve is consistent with what i described earlier — the fine-tuning optimization landscape for EfficientNet on chest X-rays is chaotic, and this shows up directly in the noisy val loss.
 
The fact that training ran for ~30 epochs tells us the model was still producing marginal improvements on validation loss throughout Phase 2. EarlyStopping correctly waited for patience=7 epochs of no improvement before stopping and restoring the best weights.
 
### ROC Curve
 
![ROC Curve](documentation/efficient_roc_curve_best.png)
*Figure 3: ROC Curve — Run 5 (AUC = 0.921)*
 
An AUC of **0.921** represents solid class separation ability. However this is notably lower than both the baseline (0.949) and DenseNet (0.974). 
 
The AUC of 0.921 also needs to be interpreted carefully in the context of EfficientNet's behavior. AUC measures ranking ability across all thresholds — a model that always predicts Pneumonia for anything above a very low threshold can still achieve reasonable AUC because it ranks Pneumonia cases higher than Normal cases on average. The high Pneumonia Recall coexisting with this moderate AUC tells us the model has genuine but limited discriminative ability — it is not completely without learning, but it is not translating that learning into balanced predictions at the default threshold.
 
### Precision-Recall Curve
 
![Precision-Recall Curve](documentation/precision_recall_curve_efficient_best.png)
*Figure 4: Precision-Recall Curve — Run 5 (PR AUC = 0.940)*
 
A PR AUC of **0.940** confirms reasonable Pneumonia detection capability. The curve maintains high precision from recall 0 to approximately 0.60, after which it begins a gradual decline, dropping more steeply above 0.80 recall. This degradation pattern is earlier and steeper than DenseNet's curve (AP = 0.983), confirming that EfficientNet's Pneumonia detection, while functional, is less precise across the full operating range.
 
Comparing directly to DenseNet:
 
```
EfficientNet PR AUC: 0.940
DenseNet AP:         0.981
```
 
The 4.1 point gap in PR AUC is meaningful and consistent with the overall performance difference between the two models.
 
### Prediction Confidence Distribution
 
![Confidence Distribution](documentation/efficient_confidence_dist_best.png)
*Figure 5: Prediction confidence distribution — Run 5*
 
This plot is the most direct visual explanation of the model's behavior. The distribution confirms the confusion matrix counts exactly:
 
**Left of threshold (predicted Normal):**
- A cluster of Normal cases near 0 — the model is confident about clearly Normal cases
- A long flat tail of Normal cases spreading across the entire 0–0.5 range and extending past the threshold toward 1.0 — these are the borderline Normal cases the model cannot confidently classify
- Only 4 Pneumonia cases fall below the threshold — confirming the near-perfect Pneumonia Recall
**Right of threshold (predicted Pneumonia):**
- A massive spike of Pneumonia cases concentrated near 1.0 — the model is extremely confident about Pneumonia predictions
- 101 Normal cases on this side — the false positives that directly explain the 43.2% false positive rate on Normal patients
The long flat tail of Normal cases is the most important visual element. It stretches from near 0 all the way to near 1.0, meaning the model assigns a wide range of probabilities to Normal cases — from highly confident Normal (near 0) to effectively treating them as Pneumonia (near 1.0). This spread reflects the model's fundamental uncertainty about Normal lung tissue.

---

## 9. Why EfficientNet Underperformed

**Reason 1 — Domain gap is more severe than expected**

EfficientNet was optimized for natural colorful photographs. Chest X-rays are grayscale, high contrast, and the diagnostically relevant features are subtle textural changes in specific anatomical locations rather than the presence of recognizable objects. The pretrained filters do not map cleanly onto what matters in X-ray diagnosis.

**Reason 2 — Sequential architecture loses fine-grained textural information**

EfficientNet's progressive feature abstraction means that by the deep layers, the low-level textural details that distinguish Normal from Pneumonia lung tissue have been compressed away. The model cannot reliably identify Normal cases because the features it needs for that decision are no longer directly accessible at the point where the classification happens.

**Reason 3 — Dataset too small to fully adapt pretrained knowledge**

5 million parameters with 4434 training images — roughly 1 image per 1127 parameters — is insufficient for the model to fully adapt its ImageNet-tuned features to the chest X-ray domain without the fine-tuning process becoming unstable.

---

## 10. Summary

| Finding | Detail |
|---|---|
| Mean test accuracy | 78.8% across 5 runs — worse than baseline (81.6%) |
| Normal Precision | 97.4% mean — near perfect but reflects conservatism (std=0.011) |
| Normal Recall | 45.6% mean — worst of all models (std=0.093) |
| Pneumonia Recall | 99.2% mean — achieved through bias not learning (std=0.004) |
| AUC (Run 5) | 0.963 — strong discriminative ability at all thresholds |
| Average Precision (Run 5) | 0.969 — excellent Pneumonia detection specifically |
| Core problem | Severe systematic Pneumonia bias — defaults to Pneumonia for all ambiguous cases |
| Clinical reliability | Low — Normal Recall ranges from 0.32 to 0.57 across runs |
| Key lesson | ImageNet pretraining does not guarantee improvement on medical imaging tasks |
| Best single run | Run 5: 83% accuracy, 57% Normal Recall, 99% Pneumonia Recall |

EfficientNet-B0 fails to deliver on the promise of transfer learning for this specific task. Its sequential architecture, domain gap, and sensitivity to fine-tuning instability combine to produce a model that is worse than the simple baseline on the metrics that matter most clinically. The high AUC and AP values confirm that the model has genuine discriminative ability — but that ability is not being translated into balanced predictions at the default threshold. This stands in direct contrast to DenseNet-121, which demonstrates that the right architectural choice can overcome these limitations.

## DenseNet-121

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras import Model, Input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)

# ── Config ────────────────────────────────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 32
SEED       = 42
TRAIN_DIR  = 'data/train'
TEST_DIR   = 'data/test'
NUM_RUNS   = 5

os.makedirs('models', exist_ok=True)

# ── Data generators (built once — reused every run) ───────────────────────────
train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    horizontal_flip        = True,
    rotation_range         = 10,
    zoom_range             = 0.1,
    width_shift_range      = 0.1,
    height_shift_range     = 0.1,
    brightness_range       = [0.8, 1.2],
    validation_split       = 0.15
)
val_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    validation_split       = 0.15
)
test_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', subset='training', shuffle=True, seed=SEED
)
val_generator = val_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', subset='validation', shuffle=False, seed=SEED
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)

# ── 5 Runs ────────────────────────────────────────────────────────────────────
for run in range(1, NUM_RUNS + 1):
    print(f"\n{'='*60}")
    print(f"  RUN {run} OF {NUM_RUNS}")
    print(f"{'='*60}\n")

    phase1_path = f'models/densenet_phase1_run{run}.h5'
    phase2_path = f'models/densenet_phase2_run{run}.h5'

    # ── Build model ───────────────────────────────────────────────
    base_model = DenseNet121(
        weights     = 'imagenet',
        include_top = False,
        input_shape = (IMG_SIZE, IMG_SIZE, 3)
    )
    base_model.trainable = False

    inputs  = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base_model(inputs, training=False)
    x       = GlobalAveragePooling2D()(x)
    x       = BatchNormalization()(x)
    x       = Dense(128, activation='relu')(x)
    x       = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model_densenet = Model(inputs, outputs)

    # ── Phase 1: Feature Extraction ───────────────────────────────
    print(f"\n── Phase 1: Feature Extraction  (Run {run}) ──")
    model_densenet.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss      = tf.keras.losses.BinaryCrossentropy(),
        metrics   = ['accuracy']
    )
    checkpoint_phase1 = ModelCheckpoint(
        phase1_path, monitor='val_loss',
        save_best_only=True, verbose=1
    )
    hist_phase1 = model_densenet.fit(
        train_generator, epochs=20,
        validation_data=val_generator,
        callbacks=[checkpoint_phase1]
    )

    # ── Phase 2: Fine-Tuning ──────────────────────────────────────
    print(f"\n── Phase 2: Fine-Tuning  (Run {run}) ──")
    model_densenet.load_weights(phase1_path)

    print(f"Total layers in backbone: {len(base_model.layers)}")
    unfreeze_from = int(len(base_model.layers) * 0.8)
    print(f"Unfreezing from layer {unfreeze_from} onwards")

    base_model.trainable = True
    for layer in base_model.layers[:unfreeze_from]:
        layer.trainable = False

    trainable     = sum([1 for l in model_densenet.layers if l.trainable])
    non_trainable = sum([1 for l in model_densenet.layers if not l.trainable])
    print(f"Trainable layers: {trainable}, Frozen layers: {non_trainable}")

    model_densenet.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss      = tf.keras.losses.BinaryCrossentropy(),
        metrics   = ['accuracy']
    )
    checkpoint_phase2 = ModelCheckpoint(
        phase2_path, monitor='val_loss',
        save_best_only=True, verbose=1
    )
    early_stop = EarlyStopping(
        monitor='val_loss', patience=7,
        restore_best_weights=True, verbose=1
    )
    hist_phase2 = model_densenet.fit(
        train_generator, epochs=30,
        validation_data=val_generator,
        callbacks=[checkpoint_phase2, early_stop]
    )

    # ── Load best phase-2 weights & evaluate ──────────────────────
    best_model = load_model(phase2_path)

    val_loss, val_acc   = best_model.evaluate(val_generator,  verbose=0)
    test_loss, test_acc = best_model.evaluate(test_generator, verbose=0)
    print(f"\nVal  — Loss: {val_loss:.4f}  |  Acc: {val_acc:.4f}")
    print(f"Test — Loss: {test_loss:.4f}  |  Acc: {test_acc:.4f}")

    # ── Predictions ───────────────────────────────────────────────
    test_generator.reset()
    y_pred_probs = best_model.predict(test_generator, verbose=1).flatten()
    y_pred       = (y_pred_probs > 0.5).astype(int)
    y_true       = test_generator.classes[:len(y_pred)]

    # ── Classification report ─────────────────────────────────────
    print(f"\n── Classification Report  (Run {run}) ──")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Pneumonia']))

    # ── Evaluation dashboard ──────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle(
        f'DenseNet-121  Run {run} — Evaluation Dashboard',
        fontsize=15, fontweight='bold'
    )

    # 1. Confusion Matrix
    ax = axes[0, 0]
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Pneumonia'],
                yticklabels=['Normal', 'Pneumonia'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title('Confusion Matrix')

    # 2. Phase 2 Loss curves
    ax = axes[0, 1]
    ax.plot(hist_phase2.history['loss'],     label='Train loss', color='teal')
    ax.plot(hist_phase2.history['val_loss'], label='Val loss',   color='orange')
    ax.set_title('Phase 2 — Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)

    # 3. Phase 2 Accuracy curves
    ax = axes[0, 2]
    ax.plot(hist_phase2.history['accuracy'],     label='Train acc', color='teal')
    ax.plot(hist_phase2.history['val_accuracy'], label='Val acc',   color='orange')
    ax.set_title('Phase 2 — Accuracy')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)

    # 4. ROC Curve
    ax = axes[1, 0]
    fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_title('ROC Curve')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.legend(); ax.grid(alpha=0.3)

    # 5. Prediction Confidence Distribution
    ax = axes[1, 1]
    ax.hist(y_pred_probs[y_true == 0], bins=30, alpha=0.6,
            color='steelblue', label='Normal')
    ax.hist(y_pred_probs[y_true == 1], bins=30, alpha=0.6,
            color='tomato',    label='Pneumonia')
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='Threshold = 0.5')
    ax.set_title('Prediction Confidence Distribution')
    ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Count')
    ax.legend(); ax.grid(alpha=0.3)

    # 6. Precision–Recall Curve
    ax = axes[1, 2]
    precision, recall, _ = precision_recall_curve(y_true, y_pred_probs)
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, color='purple', lw=2, label=f'PR AUC = {pr_auc:.3f}')
    ax.set_title('Precision–Recall Curve')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plot_path = f'evaluation_densenet_run{run}.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Plots saved → {plot_path}")

print("\n✅ All 5 DenseNet-121 runs complete.")

# Model 3 — DenseNet-121 (Fine-Tuned)
## Chest X-Ray Pneumonia Detection

---

## 1. Overview

Here i cover the training, evaluation, and analysis of the third and best performing model in this study — a fine-tuned **DenseNet-121** pretrained on ImageNet. DenseNet-121 is the architecture that Stanford's CheXNet paper used to achieve radiologist-level performance on chest X-ray diagnosis. While this study uses ImageNet pretrained weights rather than CheXNet's chest X-ray pretrained weights, the architectural advantages of DenseNet proved decisive.

The central question this model answers is: **does architectural design matter more than raw model capacity or general pretraining?**

The answer from the results is a clear yes. DenseNet-121 achieves the best performance across almost every metric in the study and does so with remarkable consistency. It wins not because it is the largest model, but because its dense connectivity architecture is fundamentally better suited to the fine-grained textural recognition required for chest X-ray diagnosis.

---

## 2. What is DenseNet-121?

DenseNet stands for **Densely Connected Convolutional Network**. The 121 refers to the total number of layers. It was introduced in 2017 and won the best paper award at CVPR, one of the most prestigious computer vision conferences.

### The key innovation — dense connectivity

In standard CNNs (including the baseline and EfficientNet), information flows sequentially:

```
Standard CNN:
Layer 1 → Layer 2 → Layer 3 → Layer 4 → output
Each layer only receives input from the layer immediately before it
Once information passes through a layer it is transformed — the original is gone
```

DenseNet completely changes this — **every layer receives direct input from all previous layers**:

```
DenseNet:
Layer 1 → Layer 2
Layer 1 ──────────→ Layer 3
Layer 1 ───────────────────→ Layer 4
Layer 2 → Layer 3
Layer 2 ──────────→ Layer 4
Layer 3 → Layer 4
```

Layer 4 has direct access to the original outputs of Layer 1, Layer 2, and Layer 3 simultaneously — not just the transformed version passed by Layer 3. Every layer has access to everything that came before it.

### Why dense connectivity matters for chest X-ray diagnosis

The difference between healthy lung tissue and early pneumonia consolidation is often a **subtle textural change** — a slight increase in opacity, a mild haziness in a specific region. These are low-level features. In a standard CNN like EfficientNet they get progressively diluted as they pass through layers of transformation. By the time you reach the deep layers, the original textural signal has been abstracted away and the model can no longer directly access what it detected early on.

In DenseNet these fine-grained textural features are preserved and directly available to every subsequent layer including the final decision layers. The model never loses access to the raw textural information that distinguishes Normal from Pneumonia. This is precisely why Stanford chose DenseNet-121 for CheXNet — chest X-ray diagnosis depends critically on preserving low-level visual information across all network depths.

### Parameter comparison

| Model | Parameters |
|---|---|
| VGG16 | 138 million |
| ResNet50 | 25 million |
| DenseNet-121 | 8 million |
| EfficientNet-B0 | 5 million |

DenseNet has more parameters than EfficientNet-B0 but achieves substantially better results. The reason is feature reuse — because features are shared rather than relearned at each layer, the network does not waste parameters reconstructing information it already has. Each layer only needs to learn what is genuinely new on top of everything it already received directly.

---

## 3. Transfer Learning — Two-Phase Approach

The same two-phase strategy used for EfficientNet was applied with adjusted hyperparameters based on lessons learned from that experiment.

### Phase 1 — Feature Extraction (20 epochs)

Five more epochs than EfficientNet (15) to give the larger DenseNet backbone more time to stabilize before fine-tuning begins.

```
DenseNet-121 backbone → FROZEN
Classification head   → TRAINED from scratch
Learning rate: 1e-3
Callbacks: ModelCheckpoint (save best val_loss)
```

### Phase 2 — Fine-Tuning (up to 30 epochs)

```
First 80% of backbone → FROZEN
Last 20% of backbone  → UNFROZEN
Classification head   → CONTINUES training
Learning rate: 1e-5
Callbacks: ModelCheckpoint + EarlyStopping(patience=7)
```

EarlyStopping with patience=7 is generous by design — stopping too early was identified as a problem in earlier experiments. This gives the model enough time to find real improvements before halting.

### Critical preprocessing detail

```python
from tensorflow.keras.applications.densenet import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input, ...
)
```

Using the wrong preprocessing produces corrupted activations throughout the network — as was demonstrated dramatically in the first EfficientNet attempt which collapsed to predicting everything as Pneumonia.

---

## 4. Architecture and Classification Head

The same classification head used for EfficientNet was kept identical to ensure fair comparison. Any performance difference between the two transfer learning models comes purely from the backbone architecture.

```
Input: 224 × 224 × 3
    ↓
DenseNet-121 backbone (pretrained on ImageNet)
    ↓
GlobalAveragePooling2D
    ↓
BatchNormalization
    ↓
Dense(128, relu)
    ↓
Dropout(0.3)
    ↓
Dense(1, sigmoid)
```

---

## 5. Training Stability — 5 Independent Runs

| Run | Accuracy | Normal Precision | Normal Recall | Pneumonia Recall | Normal F1 | Pneumonia F1 |
|---|---|---|---|---|---|---|
| Run 1 | 0.89 | 0.96 | 0.75 | 0.98 | 0.84 | 0.92 |
| Run 2 | 0.87 | 0.94 | 0.68 | 0.97 | 0.79 | 0.90 |
| Run 3 | 0.89 | 0.95 | 0.76 | 0.97 | 0.84 | 0.92 |
| Run 4 | 0.89 | 0.95 | 0.75 | 0.98 | 0.84 | 0.92 |
| Run 5 | 0.88 | 0.93 | 0.75 | 0.96 | 0.83 | 0.91 |

### Aggregate Statistics

| Metric | Mean | Median | Std | Min | Max |
|---|---|---|---|---|---|
| Accuracy | 0.884 | 0.890 | 0.008 | 0.87 | 0.89 |
| Normal Precision | 0.946 | 0.950 | 0.011 | 0.93 | 0.96 |
| Normal Recall | 0.738 | 0.750 | 0.030 | 0.68 | 0.76 |
| Pneumonia Recall | 0.972 | 0.970 | 0.008 | 0.96 | 0.98 |
| Normal F1 | 0.828 | 0.840 | 0.019 | 0.79 | 0.84 |
| Pneumonia F1 | 0.914 | 0.920 | 0.008 | 0.90 | 0.92 |

---

## 6. Analyzing Each Metric

### Accuracy: mean 0.884, std 0.008 — best and most stable

DenseNet achieves the highest mean accuracy of all three models by a significant margin:

```
Baseline:     0.816
EfficientNet: 0.788
DenseNet:     0.884
```

The std of 0.008 is the tightest of any model in the study. The range across all 5 runs is only 0.87 to 0.89 — a spread of just 2 percentage points. Compare this to the baseline's range of 0.76 to 0.87 and EfficientNet's range of 0.74 to 0.83. DenseNet produces reliable, consistent results every single run.

### Normal Precision: mean 0.946, std 0.011 — high and stable

When DenseNet predicts a patient is Normal, it is correct 94.6% of the time. This is slightly lower than EfficientNet's 97.4% — but as we established in the EfficientNet analysis, EfficientNet's higher precision reflects extreme conservatism rather than genuine strength. DenseNet's lower precision means it is less conservative and more willing to call Normal — which is exactly why its recall is so much better.

A precision of 94.6% is clinically excellent. Doctors can trust a Normal prediction from this model.

### Normal Recall: mean 0.738, std 0.030 — the most important improvement in the study

This is the number that defines DenseNet's superiority. Normal Recall of 0.738 compared to:

```
Baseline:     0.534  → DenseNet is +20.4 percentage points better
EfficientNet: 0.456  → DenseNet is +28.2 percentage points better
```

In practical terms: per 1000 patients DenseNet correctly clears 282 more healthy people than EfficientNet. These are people who would have been unnecessarily flagged, subjected to further testing, and caused unnecessary stress and healthcare cost under EfficientNet's model.

The std of 0.030 is dramatically lower than the baseline's 0.113 and EfficientNet's 0.093. DenseNet's Normal Recall never drops below 0.68 — even its worst run outperforms EfficientNet's best run (0.57). This consistency is as important as the mean improvement.

**Why does DenseNet achieve this?**

Dense connectivity builds genuine positive representations of Normal lung tissue. The model does not just ask "does this look like Pneumonia?" — it develops a rich understanding of what healthy lung tissue looks like from multiple levels of abstraction simultaneously. Every deep layer has direct access to the original fine-grained textural features that characterize Normal tissue. This allows the model to recognize Normal cases confidently across a much wider range of presentations, including borderline cases that EfficientNet would default to Pneumonia.

### Pneumonia Recall: mean 0.972, std 0.008 — the tradeoff

DenseNet catches 97.2% of Pneumonia cases. This is slightly lower than EfficientNet's 99.2% but higher than the baseline's 98.2%.

Crucially, the way DenseNet achieves its Pneumonia Recall is fundamentally different from EfficientNet. EfficientNet's high Pneumonia Recall comes from aggressive bias — defaulting to Pneumonia for everything ambiguous. DenseNet's comes from genuine learning — it has rich representations of both classes and classifies each confidently based on actual features.

The result is a model that catches 97.2% of Pneumonia cases while also correctly identifying 73.8% of Normal cases. EfficientNet catches 99.2% of Pneumonia cases while only identifying 45.6% of Normal cases. In a screening context the DenseNet operating point is far more clinically useful.

### Normal F1: mean 0.828, std 0.019 — best of all models by a wide margin

```
Baseline:     0.678
EfficientNet: 0.614
DenseNet:     0.828
```

F1 score balances precision and recall. DenseNet's Normal F1 of 0.828 is 15 points better than the baseline and 21.4 points better than EfficientNet. This is the single metric that best summarizes overall model quality for Normal detection.

### Pneumonia F1: mean 0.914, std 0.008 — best of all models

```
Baseline:     0.870
EfficientNet: 0.856
DenseNet:     0.914
```

DenseNet is not just better at Normal detection — it is also better at Pneumonia detection by every balanced measure. The high Pneumonia F1 reflects both high recall and stronger precision than EfficientNet on the Pneumonia class.

---

## 7. The Balanced Precision-Recall Profile

The contrast between DenseNet and EfficientNet's precision-recall behavior is the most analytically important finding of the study:

```
EfficientNet: Normal Precision 0.974, Normal Recall 0.456
              → Extremely conservative
              → Only calls Normal when near-certain
              → Misses 54% of healthy patients

DenseNet:     Normal Precision 0.946, Normal Recall 0.738
              → Balanced and confident
              → Correctly identifies 73.8% of healthy patients
              → When it says Normal it is right 94.6% of the time
```

DenseNet accepts slightly lower Normal Precision (0.946 vs 0.974) in exchange for dramatically higher Normal Recall (0.738 vs 0.456). This reflects a model that genuinely understands Normal lung tissue rather than one that simply avoids the Normal prediction unless overwhelmingly certain.

---

## 8. Stability — The Hidden Advantage

DenseNet is not just the best performing model — it is the most predictable:

```
Accuracy std:
Baseline:     0.042  ← high variance
EfficientNet: 0.033  ← moderate variance
DenseNet:     0.008  ← minimal variance

Normal Recall std:
Baseline:     0.113  ← extreme variance
EfficientNet: 0.093  ← high variance
DenseNet:     0.030  ← low variance
```

DenseNet's stability comes from two compounding factors. First, fixed pretrained weights eliminate initialization variance. Second, dense connectivity provides more stable gradient flow during fine-tuning — gradients can travel through multiple paths to every layer, making the learning process less sensitive to the specific ordering of batches. This is exactly the instability that plagued EfficientNet's fine-tuning process.

For a medical application where models may need to be retrained periodically on new data, knowing that you will get consistent results every time is as valuable as the performance improvement itself.

---

## 9. Detailed Evaluation — Best Run (Run 3)

The following evaluation is based on **Run 3**, the best performing run (Accuracy 0.89, Normal Recall 0.76, Pneumonia Recall 0.97).

### Classification Report (Run 3)

```
              precision    recall  f1-score   support

      Normal       0.95      0.76      0.84       234
   Pneumonia       0.87      0.97      0.92       390

    accuracy                           0.89       624
   macro avg       0.91      0.87      0.88       624
weighted avg       0.90      0.89      0.89       624
```

**Normal Recall (0.76):** The best Normal Recall achieved by any model across any run in the entire study. 76% of healthy patients correctly identified.

**Pneumonia Recall (0.97):** Catches 97 out of every 100 Pneumonia patients. Clinically excellent.

**Normal Precision (0.95):** When DenseNet predicts Normal it is correct 95% of the time.

**Accuracy (0.89):** The highest single-run accuracy in the study.

### Confusion Matrix

![Confusion Matrix](documentation/confusion_matrix_densenet_best.png)
*Figure 1: Confusion matrix — Run 3 (best run)*

| | Predicted Normal | Predicted Pneumonia |
|---|---|---|
| **Actual Normal** | 178  True Negative | 56  False Positive |
| **Actual Pneumonia** | 10  False Negative | 380  True Positive |

The **10 False Negatives** mean 10 Pneumonia patients out of 390 were missed — consistent with the 97.4% Pneumonia Recall. The **56 False Positives** represent healthy patients incorrectly flagged as sick. Compare this directly to EfficientNet's 101 false positives — DenseNet generates 45 fewer unnecessary referrals per 234 Normal patients. This single comparison encapsulates the entire story of why DenseNet is the superior clinical model.

### Training Curves — Phase 2

![Phase 2 Loss](documentation/loss_curves_densenet_best.png)
*Figure 2: Phase 2 training and validation loss curves — Run 3*

Phase 2 ran for approximately **22 epochs** before EarlyStopping triggered. The loss curves show a specific and important pattern:

```
Train loss:  0.09 → 0.04  (declining consistently and smoothly)
Val loss:    0.14 → 0.14  (flat and noisy throughout — not improving)
```

Training loss declines steadily across all 22 epochs while validation loss oscillates noisily between approximately 0.12 and 0.16 without any clear downward trend. This is the same pattern observed consistently across DenseNet runs — Phase 1 already brought the model to its performance ceiling on the validation set, and Phase 2 fine-tuning continued improving on training data without translating to validation improvement.

The noisy, flat validation loss does not mean the model is getting worse — EarlyStopping with patience=7 waited for 7 consecutive epochs of no improvement before halting and restoring the best checkpoint from the point of lowest val loss (approximately 0.117 around epoch 10–11). The final deployed model uses those best weights, not the final epoch weights.

The growing gap between train loss (0.04) and val loss (0.14) at the end of Phase 2 is a mild overfitting signature. However as with the baseline, mild late overfitting did not translate into poor test results — the model still achieved 89% test accuracy and 76% Normal Recall.

### ROC Curve

![ROC Curve](documentation/densenet_roc_curve_best.png)
*Figure 3: ROC Curve — Run 3 (AUC = 0.974)*

An AUC of **0.974** is the highest of any model in this study:

```
Baseline:     AUC = 0.949
EfficientNet: AUC = 0.921
DenseNet:     AUC = 0.974
```

The ranking here is important to note. EfficientNet falls below the baseline on AUC — confirming that its pretrained ImageNet features did not improve the underlying discriminative ability for this task. DenseNet's 0.974 represents a 3.2 point improvement over the baseline and a 5.3 point improvement over EfficientNet.

The curve hugs the top-left corner extremely tightly. At a False Positive Rate of just 0.05 — flagging only 5% of healthy patients — DenseNet already achieves approximately 0.78 True Positive Rate, catching 78% of sick patients. This is an exceptionally strong operating point that confirms genuinely rich class representations in both directions.

### Precision-Recall Curve

![Precision-Recall Curve](documentation/precision_recall_curve_densenet_best.png)
*Figure 4: Precision-Recall Curve — Run 3 (PR AUC = 0.983)*

A PR AUC of **0.983** is the highest of any model:

```
Baseline:     PR AUC = 0.966
EfficientNet: PR AUC = 0.940
DenseNet:     PR AUC = 0.983
```

The curve is visually remarkable — it stays at or extremely close to 1.0 precision from recall 0 all the way to approximately 0.80 recall, only beginning to degrade meaningfully above 0.85. This means DenseNet can catch 80% of all Pneumonia cases while maintaining essentially perfect precision — zero unnecessary false alarms up to that operating point.

For comparison, EfficientNet's curve began degrading noticeably at around 0.60 recall and showed a much steeper decline. The 4.3 point gap in PR AUC between DenseNet (0.983) and EfficientNet (0.940) confirms that DenseNet is a genuinely better Pneumonia detector — not just a more balanced one.

### Prediction Confidence Distribution

![Confidence Distribution](documentation/densenet_confidence_dist_best.png)
*Figure 5: Prediction confidence distribution — Run 3*

This distribution is the most direct visual proof of DenseNet's superiority. Reading the counts from the plot:

**Left of threshold (predicted Normal):**
- A tall spike of approximately 100 Normal cases near 0 — the model is highly confident about clearly Normal cases
- A tail that drops off quickly, much shorter and tighter than EfficientNet's long flat spread
- Only 10 Pneumonia cases fall below the threshold — confirming the 97.4% Pneumonia Recall

**Right of threshold (predicted Pneumonia):**
- A massive spike of approximately 350 Pneumonia cases concentrated near 1.0
- Only 56 Normal cases on this side — compared to EfficientNet's 101

The critical comparison is the shape of the Normal distribution. EfficientNet showed a long flat tail of Normal cases spreading all the way across the 0–1 range — the visual signature of deep uncertainty about Normal cases. DenseNet shows a much taller, tighter cluster near 0 that drops off quickly. The model is not just confident about obviously clear-cut Normal cases — it is also more confident about borderline ones, which is exactly what allows it to correctly classify 76% of Normal patients instead of EfficientNet's 57%.

This visual difference is dense connectivity made visible. Richer representations of Normal lung tissue translate directly into higher confidence predictions for Normal cases, which translates directly into the 28.2 percentage point improvement in Normal Recall over EfficientNet.

---

## 10. Why DenseNet Won

**Dense connectivity preserves fine-grained textural features**

The subtle textural differences between normal lung tissue and pneumonia consolidation require low-level feature preservation across all network depths. DenseNet's architecture guarantees this by design — every layer has direct access to every previous layer's output. EfficientNet's sequential transformation progressively abstracts this information away.

**Richer representations of both classes**

Because every deep layer has direct access to all previous representations, DenseNet builds the richest possible understanding of both Normal and Pneumonia. It does not just learn what Pneumonia looks like — it develops genuine positive understanding of what Normal lung tissue looks like from multiple scales simultaneously. This is what enables the superior Normal Recall.

**More stable fine-tuning**

Multiple gradient paths through the dense connectivity make the fine-tuning process less sensitive to batch ordering. EfficientNet showed significant run-to-run variance despite fixed pretrained weights because its sequential architecture makes the optimization landscape during fine-tuning more chaotic. DenseNet's redundant gradient paths average out this chaos, producing stable and predictable results across all 5 runs.

---

## 11. Summary

| Finding | Detail |
|---|---|
| Mean test accuracy | 88.4% across 5 runs — best of all models |
| Normal Precision | 94.6% mean — high and consistent (std=0.011) |
| Normal Recall | 73.8% mean — best of all models by a wide margin (std=0.030) |
| Pneumonia Recall | 97.2% mean — achieved through learning, not bias (std=0.008) |
| Normal F1 | 82.8% mean — best of all models (std=0.019) |
| Pneumonia F1 | 91.4% mean — best of all models (std=0.008) |
| AUC (Run 3) | 0.974 — highest of all models |
| PR AUC (Run 3) | 0.983 — highest of all models |
| Stability | Most consistent model — std of 0.008 on accuracy across 5 runs |
| Best single run | Run 3: 89% accuracy, 76% Normal Recall, 97% Pneumonia Recall |
| Key limitation | Class imbalance still suppresses Normal Recall — 26.2% of healthy patients still incorrectly flagged |

DenseNet-121 is the strongest model in this study by every measure. Its architectural design, originally chosen for CheXNet, proves its value even when using ImageNet rather than chest X-ray pretrained weights. The results confirm that for medical imaging tasks where subtle textural differences are diagnostically critical, dense connectivity is not just a nice-to-have — it is the architectural property that makes the difference between a clinically reliable model and one that is not.

# Full Model Comparison
## Chest X-Ray Pneumonia Detection Study

---

## 1. Overview

This document brings together the results of all three models trained in this study and provides a complete analytical comparison. The three models represent three fundamentally different approaches to the same binary classification task — detecting pneumonia in chest X-ray images.

| Model | Approach | Parameters | Pretraining |
|---|---|---|---|
| Baseline CNN | Trained from scratch | ~200,000 | None |
| EfficientNet-B0 | Fine-tuned | 5 million | ImageNet (1.4M natural photos) |
| DenseNet-121 | Fine-tuned | 8 million | ImageNet (1.4M natural photos) |

Each model was trained **5 independent times** to measure run-to-run variance and produce statistically representative results. All comparisons use mean values across runs — the scientifically honest approach. Single best runs are also reported for reference but should not be used as the primary comparison point.

---

## 2. The Complete Results Table

| Metric | Baseline Mean | Baseline Std | EfficientNet Mean | EfficientNet Std | DenseNet Mean | DenseNet Std |
|---|---|---|---|---|---|---|
| Accuracy | 0.816 | ±0.042 | 0.788 | ±0.033 | **0.884** | ±0.008 |
| Normal Precision | 0.952 | ±0.022 | **0.974** | ±0.011 | 0.946 | ±0.011 |
| Normal Recall | 0.534 | ±0.113 | 0.456 | ±0.093 | **0.738** | ±0.030 |
| Pneumonia Recall | 0.982 | ±0.008 | **0.992** | ±0.004 | 0.972 | ±0.008 |
| Normal F1 | 0.678 | ±0.093 | 0.614 | ±0.088 | **0.828** | ±0.019 |
| Pneumonia F1 | 0.870 | ±0.023 | 0.856 | ±0.019 | **0.914** | ±0.008 |

---

## 3. Best Run Comparison

For reference, the best single run of each model:

| Metric | Baseline Best (Run 2) | EfficientNet Best (Run 5) | DenseNet Best (Run 3) |
|---|---|---|---|
| Accuracy | 0.87 | 0.83 | **0.89** |
| Normal Recall | 0.70 | 0.57 | **0.76** |
| Pneumonia Recall | **0.97** | **0.99** | 0.97 |
| Normal F1 | 0.80 | 0.72 | **0.84** |
| Pneumonia F1 | 0.90 | 0.88 | **0.92** |

Even comparing best runs, DenseNet wins on every metric except Pneumonia Recall where EfficientNet edges it. Critically, EfficientNet's best run (0.83 accuracy) is worse than the baseline's best run (0.87), confirming that EfficientNet is the weakest model in this study across both mean and best-run comparisons.

---

## 4. The Stability Story

Standard deviation reveals how much you can trust any given training run:

```
Accuracy std:
Baseline:     ±0.042  ← high variance — lottery every run
EfficientNet: ±0.033  ← moderate variance
DenseNet:     ±0.008  ← minimal variance — always reliable

Normal Recall std:
Baseline:     ±0.113  ← extreme variance (range: 0.40 to 0.70)
EfficientNet: ±0.093  ← high variance  (range: 0.32 to 0.57)
DenseNet:     ±0.030  ← low variance   (range: 0.68 to 0.76)
```

DenseNet's minimum Normal Recall across all 5 runs (0.68) is better than EfficientNet's maximum (0.57). This means DenseNet's worst run is better than EfficientNet's best run on the most clinically important metric. This is not a marginal difference — it represents a fundamentally more reliable model.

---

## 5. The Pneumonia Recall Paradox — Why EfficientNet's High Number Is Not Good

This requires careful explanation because it is counterintuitive.

EfficientNet has the highest mean Pneumonia Recall (0.992) across all models. On the surface this sounds clinically excellent — catching 99.2% of sick patients. But the way this number is achieved reveals a serious problem.

EfficientNet achieves near-perfect Pneumonia Recall by **predicting Pneumonia for almost everything**. When a model is extremely biased toward one class, it catches almost all cases of that class — not because it learned good features, but because it almost never makes the opposite prediction.

The evidence is in the Normal Recall:

```
EfficientNet Normal Recall: 0.456
```

EfficientNet misses 54.4% of healthy patients — flagging them all as sick. A model that predicts Pneumonia for 100% of all patients would achieve 100% Pneumonia Recall and 0% Normal Recall. EfficientNet is approaching this degenerate behavior.

In a clinical context, deploying EfficientNet would mean:

```
For every 1000 patients screened:
→ 99.2% of 390 Pneumonia cases correctly identified 
→ Only 45.6% of 234 Normal patients correctly cleared 
→ 54.4% of 234 Normal patients = ~127 healthy people flagged as sick 
```

127 unnecessary referrals per 1000 patients screened is not acceptable. The high Pneumonia Recall is real but it comes at a cost that makes the model clinically unusable for Normal patients.

DenseNet's operating point is far more balanced and useful:

```
For every 1000 patients screened:
→ 97.2% of Pneumonia cases correctly identified 
→ 73.8% of Normal patients correctly cleared 
→ Only 26.2% of Normal patients flagged unnecessarily 
```

DenseNet misses 2 more Pneumonia cases per 390 compared to EfficientNet, but correctly clears 282 more healthy patients per 1000. This is a dramatically better clinical operating point.

---

## 6. Why Each Model Performed the Way It Did

### Why the baseline performed this way

The baseline has no prior knowledge. Starting from random weights with a 2.9:1 class imbalance, it learns Pneumonia features quickly and reliably while Normal detection becomes dependent on initialization luck. The result is a model that is a reliable Pneumonia detector but an unreliable Normal detector. Its fundamental limitation is not architecture — it is the combination of random initialization and class imbalance with no mechanism to overcome either.

### Why EfficientNet underperformed

EfficientNet brought pretrained knowledge from ImageNet but the domain gap between natural photos and chest X-rays was more severe than expected. Its sequential feature transformation progressively abstracts away the fine-grained textural information that distinguishes Normal from Pneumonia lung tissue. By the time information reaches the deep layers, the model has lost the low-level signal it needs for confident Normal detection. The result is a model that defaults to Pneumonia for anything ambiguous — producing near-perfect Pneumonia Recall through bias rather than learning.

Compounding this, EfficientNet's fine-tuning process proved sensitive to batch ordering, producing significant run-to-run variance despite fixed pretrained weights. This instability reflects an imperfect feature transfer — when pretrained features do not map cleanly onto the target domain, the fine-tuning optimization landscape becomes chaotic.

### Why DenseNet won

DenseNet's dense connectivity ensures that fine-grained textural features from early layers remain directly accessible to all deeper layers throughout the entire network. For chest X-ray diagnosis where the difference between Normal and Pneumonia is often a subtle low-level textural change, this architectural property is decisive.

The model builds genuine positive representations of Normal lung tissue from multiple levels of abstraction simultaneously. It does not just learn "what Pneumonia looks like" — it develops a rich understanding of what healthy lung tissue actually looks like at every scale. This is what enabled the dramatic improvement in Normal Recall and why the confidence distributions show a much tighter, more confident Normal cluster compared to EfficientNet.

DenseNet's dense connectivity also provides more stable gradient flow during fine-tuning — multiple paths for gradients to travel mean the optimization process is less sensitive to batch ordering, producing the remarkably low variance seen across all 5 runs.

---

## 7. The Ranking Is Clear

| Rank | Model | Why |
|---|---|---|
| 1st | DenseNet-121 | Best accuracy, best Normal Recall, best F1 scores, most stable, highest AUC and AP |
| 2nd | Baseline CNN | Better than EfficientNet on accuracy and Normal detection despite zero pretraining |
| 3rd | EfficientNet-B0 | Worst accuracy, worst Normal Recall, clinically unreliable Pneumonia bias |

The fact that the baseline outperforms a 5-million parameter pretrained model is one of the most important findings of this study. It confirms that pretraining is not a guaranteed improvement — domain relevance, architectural suitability, and dataset size all determine whether transfer learning benefits materialize.

---

## 8. The Persistent Challenge — Class Imbalance

Despite DenseNet's superior performance, all three models share the same fundamental limitation:

```
Mean Pneumonia Recall across all models: ~0.982
Mean Normal Recall across all models:    ~0.576

Gap: ~40.6 percentage points
```

Even DenseNet, the strongest model, only achieves 73.8% mean Normal Recall. 26.2% of healthy patients are still incorrectly flagged. This is no longer an architecture problem — DenseNet has demonstrated that with the right architecture the gap can be substantially reduced. The remaining gap is a **data problem**.

---

## 9. The Five Key Findings of This Study

**Finding 1 — Pretraining is not a guaranteed improvement**

EfficientNet, despite 5 million pretrained parameters vs the baseline's 200,000 random ones, is the worst performing model. Domain gap, architectural mismatch, and dataset size limitations can completely negate the advantages of pretrained weights.

**Finding 2 — Architecture fit is the decisive factor**

DenseNet won not because it is the biggest or most complex model but because its dense connectivity architecture matches the requirements of the task. Chest X-ray diagnosis depends on preserving fine-grained textural features — and DenseNet's architecture guarantees this by design.

**Finding 3 — High Pneumonia Recall without balanced Normal Recall is not clinical success**

EfficientNet's 99.2% Pneumonia Recall sounds excellent but is achieved through extreme Pneumonia bias that makes the model clinically unreliable. A model that catches nearly every sick patient while flagging 54% of healthy patients as sick is not fit for deployment in a screening context.

**Finding 4 — Stability is as important as peak performance**

DenseNet's std of ±0.008 on accuracy means you know with high confidence what you will get every time you train or retrain it. The baseline's ±0.042 and EfficientNet's ±0.093 on Normal Recall mean these models are lotteries. In production medical systems, reliability is not optional.

**Finding 5 — Class imbalance is the unresolved challenge**

All three models are affected by the 2.9:1 Pneumonia-to-Normal imbalance. The right architecture (DenseNet) reduces the impact significantly but does not eliminate it. Addressing the data imbalance directly is the highest-impact next step available.

---

## 10. Recommended Model for Deployment

Based on all results, **DenseNet-121 is the recommended model** with the following justification:

- Highest mean accuracy (88.4% vs 81.6% vs 78.8%)
- Highest Normal Recall (73.8%) — the most clinically important improvement
- Most stable results (std=0.008 on accuracy) — reliable and reproducible
- Most balanced precision-recall profile — clinically rational behavior
- Highest AUC (0.974) and PR AUC (0.983)
- Architecturally motivated by CheXNet, the state-of-the-art for this exact task
- Worst run (Normal Recall 0.68) is better than EfficientNet's best run (0.57)

---